# Unembed Projection Analysis

How components project through W_U: prediction trajectory,
component contributions, per-head alignment, and direction stability.

## Why This Matters

Unembedding projection examines how the model's final representation is projected into vocabulary space to produce predictions. The unembedding matrix is the lens through which all internal computation becomes observable, making its structure fundamental to interpretability.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.unembed_projection_analysis import (
    residual_unembed_trajectory, component_unembed_contribution,
    unembed_alignment_per_head, unembed_direction_stability,
    unembed_projection_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Prediction Trajectory

Top predictions at each layer through the unembedding.

In [ ]:
result = residual_unembed_trajectory(model, tokens, position=-1, top_k=3)
for p in result['per_layer']:
    toks = ', '.join(f'{t["token_id"]}({t["logit"]:.2f})' for t in p['top_tokens'])
    print(f"  Layer {p['layer']}: top={p['top_prediction']}, logit={p['top_logit']:.4f} | {toks}")

## Component Unembed Contribution

Attention vs MLP contribution to the target logit.

In [ ]:
for layer in range(4):
    result = component_unembed_contribution(model, tokens, layer=layer, position=-1)
    print(f"  Layer {layer}: attn={result['attn_contribution']:+.4f}, "
          f"mlp={result['mlp_contribution']:+.4f} [{result['dominant']}]")

## Per-Head Unembed Alignment

Which heads contribute most to the target token logit?

In [ ]:
for layer in range(4):
    result = unembed_alignment_per_head(model, tokens, layer=layer, position=-1)
    print(f"  Layer {layer} (top: H{result['top_head']}):")
    for h in result['per_head'][:3]:
        bar = '█' * int(abs(h['logit_contribution']) * 20)
        print(f"    H{h['head']}: {h['logit_contribution']:+.4f} {bar}")

## Direction Stability

How stable is alignment with the predicted unembed direction?

In [ ]:
result = unembed_direction_stability(model, tokens, position=-1)
print(f"Target: token {result['target_token']}\n")
for p in result['per_layer']:
    bar = '█' * int(abs(p['cosine_to_unembed']) * 20)
    print(f"  Layer {p['layer']}: cos={p['cosine_to_unembed']:.4f} {bar}")

## Projection Summary

In [ ]:
result = unembed_projection_summary(model, tokens, position=-1)
print(f"Final prediction: token {result['final_prediction']}\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: pred={p['top_prediction']:3d}, "
          f"logit={p['top_logit']:.4f}, cos={p['cosine_to_final_unembed']:.4f}")